# SuperPoint Dataset Creation

Builds the supervised training set for Stretcher: pairs of SuperPoint descriptors for the same
keypoint before and after a known affine deformation, together with the strain parameters that
produced it (paper, Sec. 2.3.1).

Because the deformation is applied synthetically, the correspondence between a rest keypoint and
its deformed location is exact - which is what makes supervision possible without dense expert
annotation of real deforming tissue.

Steps:
1) Pick parameters and setup
2) Loop over images and build descriptor arrays
3) Save the dataset

In [1]:
import numpy as np
import torch
from src.notebook_utils import get_best_device, save_dataset, create_dataset

/private/tmp/claude-501/-Users-constantinvonwitzleben-Documents-BWH-Stretchers-Clean/41ad82dd-b3a8-4475-9036-0bea940f8b1c/scratchpad/cleanclone/Stretchers/lightglue/lightglue.py:24: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/Users/constantinvonwitzleben/miniconda3/envs/stretcher/lib/python3.9/site-packages/ufl/__init__.py:250: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### 1) Pick parameters and setup

Set seeds, device, data root, and descriptor grid sizes.

**Source images.** `image_dir` defaults to `data/medical_deformed`, the handful of images that
ship with this repository, so the notebook runs immediately after cloning. This is a smoke
test, not the training set.

The released model (`models/stretcher_superpoint.pth`) was trained on 694 laparoscopic frames
from **Cholec80**, excluding hepatic sequences, which are held out for evaluation. Cholec80
cannot be redistributed here; request it from
[CAMMA](http://camma.u-strasbg.fr/datasets), extract frames to `data/medical_training_data`,
and point `image_dir` there.

`deformations_per_image = 125` covers the full affine grid of Sec. 2.3.1; raise `max_images`
to scale the dataset (the paper's run produced 137,200 descriptor pairs).

In [2]:
# Parameters (edit here)
image_dir = 'data/medical_deformed'   # shipped sample; see note above
max_images = 2
deformations_per_image = 125
kp_per_deformation = 1

# Reproducibility
torch.manual_seed(0)
np.random.seed(0)

# Device
device = get_best_device()
print(f'Device: {device}')

Device: mps


### 2) Loop over images and build descriptors
For each image, extract SuperPoint descriptors, then apply deformations to one keypoint per deformation, and sample the corresponding deformed descriptors. Also save the deformation that corresponds to each descriptor pair.


In [3]:
# Step 2: create dataset via helper
base_descriptors, deformed_descriptors, deformation_idx = create_dataset(
    image_dir=image_dir,
    device=device,
    deformations_per_image=deformations_per_image,
    kp_per_deformation=kp_per_deformation,
    max_images=max_images,
)


Number of images: 2


Processed 0 images


### 3) Save the dataset
Convert to torch tensors and save a .pth file with descriptors and transformations.


In [4]:
# Step 3: save dataset
output_path = 'data/SuperPoint_Descriptors_Dataset_Test.pth'
save_dataset(
    base_descriptors,
    deformed_descriptors,
    deformation_idx,
    output_path
)

Saved dataset to: data/SuperPoint_Descriptors_Dataset_Test.pth


/private/tmp/claude-501/-Users-constantinvonwitzleben-Documents-BWH-Stretchers-Clean/41ad82dd-b3a8-4475-9036-0bea940f8b1c/scratchpad/cleanclone/Stretchers/src/notebook_utils.py:343: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch_deformations = torch.tensor(deformation_idx)
